In [1]:
!pip install geoopt

In [2]:
import matplotlib.pyplot as plt
import geoopt
import torch
import itertools
import torch.nn as nn
import numpy as np
import tqdm
import seaborn as sns
from ipywidgets import interact
from geoopt.manifolds.stereographic import StereographicExact
%matplotlib inline

In [3]:
class COLORS:
    SHINY_GREEN = "#bfffbf"
    SHINY_BLUE = "#a9e7ff"
    MAT_RED = "#ffc7c7"
    MAT_YELLOW = "#ffffbd"
    NEON_PINK = "#ff3ea0"
    BACKGROUND_BLUE = "#1e0c45"
    TEXT_COLOR = "#ffffff"

In [4]:
def add_geodesic_grid(ax: plt.Axes, manifold: geoopt.Stereographic, line_width=0.1):

    # define geodesic grid parameters
    N_EVALS_PER_GEODESIC = 10000
    STYLE = "--"
    COLOR = "gray"
    LINE_WIDTH = line_width

    # get manifold properties
    K = manifold.k.item()
    R = manifold.radius.item()

    # get maximal numerical distance to origin on manifold
    if K < 0:
        # create point on R
        r = torch.tensor((R, 0.0), dtype=manifold.dtype)
        # project point on R into valid range (epsilon border)
        r = manifold.projx(r)
        # determine distance from origin
        max_dist_0 = manifold.dist0(r).item()
    else:
        max_dist_0 = np.pi * R
    # adjust line interval for spherical geometry
    circumference = 2*np.pi*R

    # determine reasonable number of geodesics
    # choose the grid interval size always as if we'd be in spherical
    # geometry, such that the grid interpolates smoothly and evenly
    # divides the sphere circumference
    n_geodesics_per_circumference = 4 * 6  # multiple of 4!
    n_geodesics_per_quadrant = n_geodesics_per_circumference // 2
    grid_interval_size = circumference / n_geodesics_per_circumference
    if K < 0:
        n_geodesics_per_quadrant = int(max_dist_0 / grid_interval_size)

    # create time evaluation array for geodesics
    if K < 0:
        min_t = -1.2*max_dist_0
    else:
        min_t = -circumference/2.0
    t = torch.linspace(min_t, -min_t, N_EVALS_PER_GEODESIC)[:, None]

    # define a function to plot the geodesics
    def plot_geodesic(gv):
        ax.plot(*gv.t().numpy(), STYLE, color=COLOR, linewidth=LINE_WIDTH)

    # define geodesic directions
    u_x = torch.tensor((0.0, 1.0))
    u_y = torch.tensor((1.0, 0.0))

    # add origin x/y-crosshair
    o = torch.tensor((0.0, 0.0))
    if K < 0:
        x_geodesic = manifold.geodesic_unit(t, o, u_x)
        y_geodesic = manifold.geodesic_unit(t, o, u_y)
        plot_geodesic(x_geodesic)
        plot_geodesic(y_geodesic)
    else:
        # add the crosshair manually for the sproj of sphere
        # because the lines tend to get thicker if plotted
        # as done for K<0
        ax.axvline(0, linestyle=STYLE, color=COLOR, linewidth=LINE_WIDTH)
        ax.axhline(0, linestyle=STYLE, color=COLOR, linewidth=LINE_WIDTH)

    # add geodesics per quadrant
    for i in range(1, n_geodesics_per_quadrant):
        i = torch.as_tensor(float(i))
        # determine start of geodesic on x/y-crosshair
        x = manifold.geodesic_unit(i*grid_interval_size, o, u_y)
        y = manifold.geodesic_unit(i*grid_interval_size, o, u_x)

        # compute point on geodesics
        x_geodesic = manifold.geodesic_unit(t, x, u_x)
        y_geodesic = manifold.geodesic_unit(t, y, u_y)

        # plot geodesics
        plot_geodesic(x_geodesic)
        plot_geodesic(y_geodesic)
        if K < 0:
            plot_geodesic(-x_geodesic)
            plot_geodesic(-y_geodesic)

In [5]:
def setup_plot(manifold, lo=None, width=7, height=7, grid_line_width=0.3, with_background=True):

    # create figure
    fig = plt.figure()

    # determine manifold properties
    K = manifold.k
    R = manifold.radius

    # add circle
    circle = plt.Circle((0, 0), R, fill=with_background, color=COLORS.BACKGROUND_BLUE)
    plt.gca().add_artist(circle)
    if K > 0:
        circle_border = plt.Circle((0, 0), R, fill=False, color="gray",
                                   linewidth=2.0)
        plt.gca().add_artist(circle_border)


    # add background color
    if not(K < 0) and with_background:
        plt.gca().set_facecolor(COLORS.BACKGROUND_BLUE)

    # set up plot axes and aspect ratio
    if lo==None:
        if K < 0:
            lo = -R - 0.1
        else:
            lo = -2 * R - 0.1
    hi = -lo
    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.gca().set_aspect("equal")

    # add grid of geodesics
    add_geodesic_grid(plt.gca(), manifold, line_width=grid_line_width)

    return fig, plt, (lo, hi)

In [14]:
# create manifold with initial k=-1.0 (Poincaré Ball)
manifold = StereographicExact(k=-1.0, learnable=False)

# set to double precision for numerical stability
manifold = manifold.to(dtype=torch.float64)

def show_plot(k):
  manifold.k.data = torch.tensor(k, dtype=torch.float64)
  fig, plt, (lo, hi) = setup_plot(manifold, lo = -3)
  plt.show()

interact(show_plot, k=(-1, 1, .01));

interactive(children=(FloatSlider(value=0.0, description='k', max=1.0, min=-1.0, step=0.01), Output()), _dom_c…